# P10.6-AI - RSNA dataset preflight

Preflight train-only para RSNA LumbarDISC. Este notebook prepara inventario, distribuciones y reportes sanitizados para clasificacion asistida de hallazgo candidato. No entrena, no accede al test oficial y no genera diagnostico clinico.

## Guardias operativas

- `humanReviewRequired=true` y `notClinicalDiagnosis=true`.
- El conjunto oficial de test se ignora completamente si existe.
- Protrusion y extrusion no se entrenan con RSNA.
- Los reportes se escriben fuera de Git bajo `PFI_P10_6_OUTPUT_ROOT`.

In [2]:
# ============================================================
# P10.6-AI — Descarga inicial de RSNA train-only en Google Drive
# Ejecutar una sola vez antes del preflight real del Notebook 53.
#
# Seguridad:
# - No guarda el token en Drive.
# - No extrae test_images.
# - No extrae test_series_descriptions.csv.
# - No extrae sample_submission.csv.
# - No modifica archivos originales si ya están completos.
# ============================================================

from __future__ import annotations

import getpass
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path, PurePosixPath

# ------------------------------------------------------------
# 1. Montar Google Drive
# ------------------------------------------------------------

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError(
        "Esta celda debe ejecutarse dentro de Google Colab."
    ) from exc

drive.mount("/content/drive", force_remount=False)


# ------------------------------------------------------------
# 2. Configuración
# ------------------------------------------------------------

COMPETITION = "rsna-2024-lumbar-spine-degenerative-classification"

PFI_ROOT = Path("/content/drive/MyDrive/PFI_MVP")
RSNA_ROOT = PFI_ROOT / "data" / "RSNA_LUMBAR_DISC"
TRAIN_IMAGES = RSNA_ROOT / "train_images"

OUTPUT_ROOT = PFI_ROOT / "results" / "P10_6_rsna_findings"
MODEL_ROOT = PFI_ROOT / "models" / "P10_6_rsna_findings"

# La descarga comprimida se mantiene temporalmente en el disco de Colab,
# no en Drive, para no ocupar el doble de almacenamiento permanente.
TEMP_DOWNLOAD_ROOT = Path("/content/rsna_kaggle_download")

REQUIRED_FILES = (
    RSNA_ROOT / "train.csv",
    RSNA_ROOT / "train_label_coordinates.csv",
    RSNA_ROOT / "train_series_descriptions.csv",
)

ALLOWED_CSV_NAMES = {
    "train.csv",
    "train_label_coordinates.csv",
    "train_series_descriptions.csv",
}

# Cambiar manualmente a True solamente si se necesita repetir toda la descarga.
FORCE_REDOWNLOAD = False

RSNA_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 3. Estado de almacenamiento
# ------------------------------------------------------------

def free_gib(path: Path) -> float:
    usage = shutil.disk_usage(path)
    return usage.free / (1024 ** 3)


print("Espacio libre aproximado:")
print(f"- Runtime Colab: {free_gib(Path('/content')):.2f} GiB")
print(f"- Google Drive:  {free_gib(Path('/content/drive/MyDrive')):.2f} GiB")
print(f"- Destino lógico: {RSNA_ROOT}")


# ------------------------------------------------------------
# 4. Comprobar si ya está descargado
# ------------------------------------------------------------

def dataset_train_is_ready() -> bool:
    csv_ready = all(path.is_file() and path.stat().st_size > 0 for path in REQUIRED_FILES)

    if not TRAIN_IMAGES.is_dir():
        return False

    # No recorre todo el dataset para esta comprobación rápida.
    first_dicom = next(TRAIN_IMAGES.rglob("*.dcm"), None)
    return csv_ready and first_dicom is not None


if dataset_train_is_ready() and not FORCE_REDOWNLOAD:
    print("\nRSNA train-only ya se encuentra disponible.")
    print("Se omite la descarga para evitar duplicados.")

else:
    # --------------------------------------------------------
    # 5. Instalar o actualizar Kaggle CLI
    # --------------------------------------------------------

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--upgrade",
            "kaggle",
        ]
    )

    kaggle_executable = shutil.which("kaggle")

    if kaggle_executable is None:
        possible_executable = Path(sys.executable).parent / "kaggle"
        if not possible_executable.exists():
            raise RuntimeError(
                "Kaggle CLI se instaló, pero no se encontró su ejecutable."
            )
        kaggle_executable = str(possible_executable)

    # --------------------------------------------------------
    # 6. Autenticación temporal
    # --------------------------------------------------------

    # Copiar únicamente el token generado en Kaggle Settings → API.
    # getpass no muestra el valor ingresado.
    kaggle_token = os.getenv("KAGGLE_API_TOKEN", "").strip()

    if not kaggle_token:
        kaggle_token = getpass.getpass(
            "Pegá tu KAGGLE_API_TOKEN y presioná Enter "
            "(el valor no se mostrará): "
        ).strip()

    if not kaggle_token:
        raise RuntimeError("No se ingresó un token de Kaggle.")

    os.environ["KAGGLE_API_TOKEN"] = kaggle_token

    TEMP_DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)

    # --------------------------------------------------------
    # 7. Verificar acceso a la competencia
    # --------------------------------------------------------

    print("\nVerificando acceso a la competencia...")

    try:
        subprocess.check_call(
            [
                kaggle_executable,
                "competitions",
                "files",
                COMPETITION,
                "--page-size",
                "20",
                "--quiet",
            ]
        )
    except subprocess.CalledProcessError as exc:
        os.environ.pop("KAGGLE_API_TOKEN", None)
        kaggle_token = ""

        raise RuntimeError(
            "Kaggle rechazó el acceso. Verificá que:\n"
            "1. hayas aceptado las reglas de la competencia;\n"
            "2. el token sea válido;\n"
            "3. la cuenta de Kaggle esté verificada."
        ) from exc

    # --------------------------------------------------------
    # 8. Descargar el paquete oficial
    # --------------------------------------------------------

    print("\nDescargando RSNA desde Kaggle...")
    print("La descarga puede demorar varios minutos.")

    try:
        subprocess.check_call(
            [
                kaggle_executable,
                "competitions",
                "download",
                COMPETITION,
                "--path",
                str(TEMP_DOWNLOAD_ROOT),
                "--force",
            ]
        )
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            "La descarga de RSNA falló. "
            "Revisá el espacio disponible y el acceso a la competencia."
        ) from exc
    finally:
        # El token deja de estar disponible para las celdas posteriores.
        os.environ.pop("KAGGLE_API_TOKEN", None)
        kaggle_token = ""

    # --------------------------------------------------------
    # 9. Resolver rutas permitidas dentro del ZIP
    # --------------------------------------------------------

    def allowed_relative_path(member_name: str) -> Path | None:
        """
        Devuelve una ruta de destino solo para archivos train autorizados.
        Nunca devuelve rutas de test o sample_submission.
        """
        normalized = PurePosixPath(member_name)

        if normalized.is_absolute() or ".." in normalized.parts:
            raise RuntimeError(
                f"Ruta insegura encontrada dentro del ZIP: {member_name!r}"
            )

        # CSV de entrenamiento permitidos.
        if normalized.name in ALLOWED_CSV_NAMES:
            return Path(normalized.name)

        # Imágenes de entrenamiento únicamente.
        if "train_images" in normalized.parts:
            index = normalized.parts.index("train_images")
            return Path(*normalized.parts[index:])

        return None


    def safe_extract_train_only(archive_path: Path) -> int:
        extracted = 0
        root_resolved = RSNA_ROOT.resolve()

        print(f"\nExtrayendo train-only desde: {archive_path.name}")

        with zipfile.ZipFile(archive_path, "r") as archive:
            for member in archive.infolist():
                if member.is_dir():
                    continue

                relative_path = allowed_relative_path(member.filename)

                if relative_path is None:
                    # test_images, test CSV y otros archivos se ignoran.
                    continue

                destination = (RSNA_ROOT / relative_path).resolve()

                if destination != root_resolved and root_resolved not in destination.parents:
                    raise RuntimeError(
                        f"Intento de extracción fuera del destino: {member.filename!r}"
                    )

                if (
                    destination.exists()
                    and destination.stat().st_size == member.file_size
                    and not FORCE_REDOWNLOAD
                ):
                    continue

                destination.parent.mkdir(parents=True, exist_ok=True)

                with archive.open(member, "r") as source:
                    with destination.open("wb") as target:
                        shutil.copyfileobj(
                            source,
                            target,
                            length=8 * 1024 * 1024,
                        )

                extracted += 1

                if extracted % 5000 == 0:
                    print(f"Archivos extraídos: {extracted}")

        return extracted


    # --------------------------------------------------------
    # 10. Extraer únicamente train
    # --------------------------------------------------------

    archives = sorted(TEMP_DOWNLOAD_ROOT.glob("*.zip"))

    if not archives:
        raise RuntimeError(
            f"No se encontró ningún ZIP descargado en {TEMP_DOWNLOAD_ROOT}."
        )

    extracted_total = 0

    for archive_path in archives:
        extracted_total += safe_extract_train_only(archive_path)

    print(f"\nArchivos nuevos extraídos: {extracted_total}")

    # --------------------------------------------------------
    # 11. Copiar archivos directos si Kaggle los entregó fuera del ZIP
    # --------------------------------------------------------

    for csv_name in ALLOWED_CSV_NAMES:
        source = TEMP_DOWNLOAD_ROOT / csv_name
        destination = RSNA_ROOT / csv_name

        if source.is_file() and (
            FORCE_REDOWNLOAD or not destination.is_file()
        ):
            shutil.copy2(source, destination)

    # --------------------------------------------------------
    # 12. Eliminar descarga temporal
    # --------------------------------------------------------

    shutil.rmtree(TEMP_DOWNLOAD_ROOT, ignore_errors=True)
    print("Archivos comprimidos temporales eliminados.")


# ------------------------------------------------------------
# 13. Validación final train-only
# ------------------------------------------------------------

missing_paths = [
    str(path)
    for path in REQUIRED_FILES
    if not path.is_file() or path.stat().st_size == 0
]

if not TRAIN_IMAGES.is_dir():
    missing_paths.append(str(TRAIN_IMAGES))

first_dicom = (
    next(TRAIN_IMAGES.rglob("*.dcm"), None)
    if TRAIN_IMAGES.is_dir()
    else None
)

if first_dicom is None:
    missing_paths.append(f"{TRAIN_IMAGES}/**/*.dcm")

if missing_paths:
    raise RuntimeError(
        "La descarga o extracción quedó incompleta.\nFaltan:\n- "
        + "\n- ".join(missing_paths)
    )

# La carpeta creada no contiene test oficial.
for forbidden_name in (
    "test_images",
    "test_series_descriptions.csv",
    "sample_submission.csv",
):
    forbidden_path = RSNA_ROOT / forbidden_name

    if forbidden_path.exists():
        raise RuntimeError(
            f"Se encontró contenido de test no permitido: {forbidden_path}"
        )


# ------------------------------------------------------------
# 14. Variables esperadas por Notebook 53
# ------------------------------------------------------------

os.environ["PFI_ROOT"] = str(PFI_ROOT)
os.environ["PFI_RSNA_ROOT"] = str(RSNA_ROOT)
os.environ["PFI_RSNA_TRAIN_IMAGES"] = str(TRAIN_IMAGES)
os.environ["PFI_P10_6_OUTPUT_ROOT"] = str(OUTPUT_ROOT)
os.environ["PFI_P10_6_MODEL_ROOT"] = str(MODEL_ROOT)

os.environ["PFI_RSNA_PREFLIGHT_SYNTHETIC"] = "0"
os.environ["PFI_USE_GOOGLE_DRIVE"] = "1"

print("\nConfiguración RSNA lista:")
print(f"- train.csv:                      {(RSNA_ROOT / 'train.csv').exists()}")
print(f"- train_label_coordinates.csv:    {(RSNA_ROOT / 'train_label_coordinates.csv').exists()}")
print(f"- train_series_descriptions.csv:  {(RSNA_ROOT / 'train_series_descriptions.csv').exists()}")
print(f"- train_images:                   {TRAIN_IMAGES.exists()}")
print(f"- Primer DICOM encontrado:        {first_dicom is not None}")
print("- officialTestPresent:            false")
print("- officialTestAccessed:           false")
print("- syntheticMode:                  false")
print("\nYa podés continuar con las celdas originales del Notebook 53.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Espacio libre aproximado:
- Runtime Colab: 87.72 GiB
- Google Drive:  83.33 GiB
- Destino lógico: /content/drive/MyDrive/PFI_MVP/data/RSNA_LUMBAR_DISC
Pegá tu KAGGLE_API_TOKEN y presioná Enter (el valor no se mostrará): ··········

Verificando acceso a la competencia...

Descargando RSNA desde Kaggle...
La descarga puede demorar varios minutos.

Extrayendo train-only desde: rsna-2024-lumbar-spine-degenerative-classification.zip
Archivos extraídos: 5000
Archivos extraídos: 10000
Archivos extraídos: 15000
Archivos extraídos: 20000
Archivos extraídos: 25000
Archivos extraídos: 30000
Archivos extraídos: 35000
Archivos extraídos: 40000
Archivos extraídos: 45000
Archivos extraídos: 50000
Archivos extraídos: 55000
Archivos extraídos: 60000
Archivos extraídos: 65000
Archivos extraídos: 70000
Archivos extraídos: 75000
Archivos extraídos: 80000
Archivos extraídos: 8500

In [3]:
!pip install -q monai timm pydicom SimpleITK nbformat pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 9.4 MB/s eta 0:00:00


In [4]:
os.environ["PFI_ALLOW_REPO_CLONE"] = "1"

In [5]:
# 1) Dependencias
from __future__ import annotations

import importlib.util
import subprocess
import sys

REQUIRED_MODULES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "pydicom": "pydicom",
    "SimpleITK": "SimpleITK",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "torch": "torch",
    "torchvision": "torchvision",
    "timm": "timm",
    "monai": "monai",
    "yaml": "pyyaml",
}

missing = [package for module, package in REQUIRED_MODULES.items() if importlib.util.find_spec(module) is None]
if missing:
    try:
        import google.colab  # type: ignore  # noqa: F401
    except Exception as exc:
        raise RuntimeError(f"Dependencias faltantes fuera de Colab: {missing}") from exc
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])


In [6]:
import os

os.environ["PFI_ROOT"] = "/content/drive/MyDrive/PFI_MVP"

os.environ["PFI_RSNA_ROOT"] = (
    "/content/drive/MyDrive/PFI_MVP/data/RSNA_LUMBAR_DISC"
)

os.environ["PFI_RSNA_TRAIN_IMAGES"] = (
    "/content/drive/MyDrive/PFI_MVP/data/RSNA_LUMBAR_DISC/train_images"
)

os.environ["PFI_P10_6_OUTPUT_ROOT"] = (
    "/content/drive/MyDrive/PFI_MVP/results/P10_6_rsna_findings"
)

os.environ["PFI_P10_6_MODEL_ROOT"] = (
    "/content/drive/MyDrive/PFI_MVP/models/P10_6_rsna_findings"
)

os.environ["PFI_RSNA_PREFLIGHT_SYNTHETIC"] = "0"

In [7]:
# 2) Imports, versiones y semillas
import json
import os
from pathlib import Path

import pandas as pd
import pydicom
import SimpleITK as sitk
import torch
import yaml

from sklearn.model_selection import StratifiedGroupKFold  # noqa: F401

print({
    "python": sys.version.split()[0],
    "pytorch": torch.__version__,
    "cudaAvailable": torch.cuda.is_available(),
    "cudaVersion": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "pydicom": pydicom.__version__,
    "SimpleITK": sitk.Version_VersionString(),
})


{'python': '3.12.13', 'pytorch': '2.11.0+cpu', 'cudaAvailable': False, 'cudaVersion': None, 'gpu': None, 'pydicom': '3.0.2', 'SimpleITK': '2.5.6'}


In [8]:
# 3) Montaje opcional de Google Drive
USE_GOOGLE_DRIVE = os.getenv("PFI_USE_GOOGLE_DRIVE", "1").strip() == "1"
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive  # type: ignore

        drive.mount("/content/drive")
    except Exception:
        print("Google Drive no disponible; continuo con rutas locales/configuradas.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
# 4) Repositorio y modulo AI
os.environ["PFI_ALLOW_REPO_CLONE"] = "1"

PFI_REPO_URL = "https://github.com/EnzoAA004/PFI_MVPTest_Enzo_AImodule.git"
PFI_REPO_ROOT = Path(os.getenv(
    "PFI_REPO_ROOT",
    "/content/PFI_MVPTest_Enzo_AImodule"
))
PFI_REPO_REF = os.getenv(
    "PFI_REPO_REF",
    "enzo/p10-6-ai-rsna-findings"
)

if not (PFI_REPO_ROOT / "ai_service" / "pfi_ai_service").exists():
    subprocess.check_call(
        ["git", "clone", PFI_REPO_URL, str(PFI_REPO_ROOT)]
    )

subprocess.check_call(
    ["git", "fetch", "origin"],
    cwd=PFI_REPO_ROOT
)

subprocess.check_call(
    ["git", "checkout", PFI_REPO_REF],
    cwd=PFI_REPO_ROOT
)

sys.path.insert(
    0,
    str(PFI_REPO_ROOT / "ai_service")
)

from pfi_ai_service.training.rsna_preflight import (
    build_config,
    run_preflight,
    runtime_versions,
    set_reproducible_seed,
    validate_dataset_structure,
)

In [10]:
# 5) Configuracion reproducible
CFG = build_config()
set_reproducible_seed(CFG.seed)
print(json.dumps({
    "seed": CFG.seed,
    "syntheticMode": CFG.synthetic,
    "rsnaRootConfigured": str(CFG.rsna_root),
    "outputRootConfigured": str(CFG.output_root),
    "modelRootConfigured": str(CFG.model_root),
    "humanReviewRequired": True,
    "notClinicalDiagnosis": True,
}, indent=2))


{
  "seed": 2026,
  "syntheticMode": false,
  "rsnaRootConfigured": "/content/drive/MyDrive/PFI_MVP/data/RSNA_LUMBAR_DISC",
  "outputRootConfigured": "/content/drive/MyDrive/PFI_MVP/results/P10_6_rsna_findings",
  "modelRootConfigured": "/content/drive/MyDrive/PFI_MVP/models/P10_6_rsna_findings",
  "humanReviewRequired": true,
  "notClinicalDiagnosis": true
}


In [11]:
# 6) Preflight de estructura train-only
if not CFG.synthetic:
    structure = validate_dataset_structure(CFG)
else:
    structure = {"syntheticMode": True, "officialTestAccessed": False}

if structure.get("officialTestAccessed") is not False:
    raise RuntimeError("El preflight no puede acceder al test oficial.")

print(json.dumps(structure, indent=2))


{
  "rsnaRoot": "/content/drive/MyDrive/PFI_MVP/data/RSNA_LUMBAR_DISC",
  "trainImagesRoot": "/content/drive/MyDrive/PFI_MVP/data/RSNA_LUMBAR_DISC/train_images",
  "requiredPresent": [
    "train.csv",
    "train_images",
    "train_label_coordinates.csv",
    "train_series_descriptions.csv"
  ],
  "officialTestPresent": false,
  "officialTestAccessed": false
}


In [17]:
import shutil

total, used, free = shutil.disk_usage("/content")

print({
    "totalGiB": round(total / 1024**3, 2),
    "usedGiB": round(used / 1024**3, 2),
    "freeGiB": round(free / 1024**3, 2),
})

{'totalGiB': 107.72, 'usedGiB': 38.03, 'freeGiB': 69.67}


In [18]:
# ============================================================
# P10.6 — Preparar copia local optimizada para el preflight
#
# Copia train-only desde Google Drive al disco local de Colab.
# No copia test oficial.
# Los reportes siguen escribiéndose en Google Drive.
# ============================================================

from pathlib import Path
import os
import shutil
import subprocess
import time

DRIVE_RSNA_ROOT = Path(
    "/content/drive/MyDrive/PFI_MVP/data/RSNA_LUMBAR_DISC"
)

LOCAL_RSNA_ROOT = Path(
    "/content/RSNA_LUMBAR_DISC"
)

DRIVE_OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/PFI_MVP/results/P10_6_rsna_findings"
)

DRIVE_MODEL_ROOT = Path(
    "/content/drive/MyDrive/PFI_MVP/models/P10_6_rsna_findings"
)

required_csvs = [
    "train.csv",
    "train_label_coordinates.csv",
    "train_series_descriptions.csv",
]

for name in required_csvs:
    source = DRIVE_RSNA_ROOT / name
    if not source.is_file():
        raise FileNotFoundError(f"Falta el archivo requerido: {source}")

if not (DRIVE_RSNA_ROOT / "train_images").is_dir():
    raise FileNotFoundError(
        f"Falta train_images: {DRIVE_RSNA_ROOT / 'train_images'}"
    )

if (DRIVE_RSNA_ROOT / "test_images").exists():
    raise RuntimeError(
        "Se encontró test_images en el root. "
        "Esta copia optimizada solo admite train-only."
    )

LOCAL_RSNA_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_MODEL_ROOT.mkdir(parents=True, exist_ok=True)

start = time.time()

# Copiar CSV pequeños.
for name in required_csvs:
    shutil.copy2(
        DRIVE_RSNA_ROOT / name,
        LOCAL_RSNA_ROOT / name,
    )

# Copiar DICOM con rsync.
# --ignore-existing permite reanudar si se corta.
subprocess.check_call([
    "rsync",
    "-a",
    "--ignore-existing",
    "--info=progress2",
    str(DRIVE_RSNA_ROOT / "train_images") + "/",
    str(LOCAL_RSNA_ROOT / "train_images") + "/",
])

elapsed = time.time() - start

print(f"\nCopia local completada en {elapsed / 60:.1f} minutos.")
print(f"Root local: {LOCAL_RSNA_ROOT}")
print(f"Train images local: {LOCAL_RSNA_ROOT / 'train_images'}")

# Reconfigurar solo la entrada hacia almacenamiento local.
# Los outputs continúan en Drive.
os.environ["PFI_RSNA_ROOT"] = str(LOCAL_RSNA_ROOT)
os.environ["PFI_RSNA_TRAIN_IMAGES"] = str(
    LOCAL_RSNA_ROOT / "train_images"
)
os.environ["PFI_P10_6_OUTPUT_ROOT"] = str(DRIVE_OUTPUT_ROOT)
os.environ["PFI_P10_6_MODEL_ROOT"] = str(DRIVE_MODEL_ROOT)
os.environ["PFI_RSNA_PREFLIGHT_SYNTHETIC"] = "0"
os.environ["PFI_RSNA_HASH_DICOM_OPT_IN"] = "0"

print("\nVariables actualizadas:")
print("PFI_RSNA_ROOT =", os.environ["PFI_RSNA_ROOT"])
print(
    "PFI_RSNA_TRAIN_IMAGES =",
    os.environ["PFI_RSNA_TRAIN_IMAGES"],
)
print(
    "PFI_P10_6_OUTPUT_ROOT =",
    os.environ["PFI_P10_6_OUTPUT_ROOT"],
)

KeyboardInterrupt: 

In [19]:
from pathlib import Path
import shutil

local_train = Path("/content/RSNA_LUMBAR_DISC/train_images")

count = sum(1 for _ in local_train.rglob("*.dcm")) if local_train.exists() else 0
size_bytes = sum(
    p.stat().st_size
    for p in local_train.rglob("*.dcm")
) if local_train.exists() else 0

total, used, free = shutil.disk_usage("/content")

print({
    "localDicomCount": count,
    "localSizeGiB": round(size_bytes / 1024**3, 2),
    "freeGiB": round(free / 1024**3, 2),
})

{'localDicomCount': 13695, 'localSizeGiB': 3.11, 'freeGiB': 67.43}


In [ ]:
CFG = build_config()
set_reproducible_seed(CFG.seed)

print(json.dumps({
    "syntheticMode": CFG.synthetic,
    "rsnaRootConfigured": str(CFG.rsna_root),
    "trainImagesConfigured": str(CFG.train_images),
    "outputRootConfigured": str(CFG.output_root),
}, indent=2))

In [ ]:
structure = validate_dataset_structure(CFG)
print(json.dumps(structure, indent=2))

In [ ]:
# 8) Evidencia sanitizada para Notebook 54
print(json.dumps({
    "dataset": summary["dataset"],
    "csvSha256": summary["csvSha256"],
    "sequenceAvailability": summary["sequenceAvailability"],
    "coordinateIssues": summary["coordinateIssues"],
    "futureMetrics": summary["futureMetrics"],
    "limitations": summary["limitations"],
    "nextNotebook": "54_internal_split_and_model_plan",
}, indent=2))
